In [1]:
!pip install -q textstat transformers torch numpy pandas tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 3.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 32.2 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 97.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import torch
import textstat
from tqdm import tqdm
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import warnings

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("⚠️ WARNING: You are using CPU. Turn on GPU in Kaggle settings for 50x speedup!")

2025-11-26 06:00:11.354723: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764136811.529757      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764136811.577086      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Using device: cuda


In [4]:
# Load your dataset
# REPLACE with your actual file path from the 'Input' section in Kaggle
df = pd.read_csv('/kaggle/input/hc3-project-cleaned-essays/filtered_essay_corpus.csv') 

# Quick check
print(f"Dataset shape: {df.shape}")
print(df.head())

Dataset shape: (290888, 3)
                                                text  generated source
0  He couldn't remember how many nights is it bee...          0  human
1  BARACK OBAMA "I am not proposing a big governm...          1     ai
2  Image copyright Getty Images Image caption Env...          0  human
3  Image caption The attack happened in Daburah, ...          1     ai
4  By If you remember from a few months ago, we w...          1     ai


In [5]:
class FeatureExtractor:
    def __init__(self, device='cuda'):
        self.device = device
        # Load GPT-2 for Perplexity (Standard Baseline)
        print("Loading GPT-2 model...")
        self.model_id = 'gpt2'
        self.model = GPT2LMHeadModel.from_pretrained(self.model_id).to(device)
        self.tokenizer = GPT2TokenizerFast.from_pretrained(self.model_id)
        
    def calculate_perplexity(self, text):
        """
        Calculates perplexity using GPT-2 with a sliding window.
        Low Perplexity = Likely AI. High Perplexity = Likely Human.
        """
        encodings = self.tokenizer(text, return_tensors='pt')
        max_length = self.model.config.n_positions
        stride = 512
        seq_len = encodings.input_ids.size(1)

        nll = []
        prev_end_loc = 0

        for begin_loc in range(0, seq_len, stride):
            end_loc = min(begin_loc + max_length, seq_len)
            trg_len = end_loc - prev_end_loc
            input_ids = encodings.input_ids[:, begin_loc:end_loc].to(self.device)
            target_ids = input_ids.clone()
            target_ids[:, :-trg_len] = -100

            with torch.no_grad():
                outputs = self.model(input_ids, labels=target_ids)
                nll.append(outputs.loss)

            prev_end_loc = end_loc
            if end_loc == seq_len:
                break

        if not nll: return np.nan
        ppl = torch.exp(torch.stack(nll).mean())
        return ppl.item()

    def calculate_burstiness(self, text):
        """
        Burstiness proxy: Standard Deviation of sentence lengths.
        AI text often has very uniform sentence lengths (Low Burstiness).
        """
        sentences = text.split('.')
        lengths = [len(s.split()) for s in sentences if len(s.split()) > 0]
        if not lengths: return 0
        return np.std(lengths)

    def extract_features(self, text):
        try:
            # 1. Perplexity (The Heavy Lifter)
            ppl = self.calculate_perplexity(text)
            
            # 2. Readability Metrics
            flesch = textstat.flesch_reading_ease(text)
            readability_grade = textstat.flesch_kincaid_grade(text)
            
            # 3. Stylistic/Statistical
            burstiness = self.calculate_burstiness(text)
            word_count = textstat.lexicon_count(text, removepunct=True)
            avg_sentence_len = textstat.avg_sentence_length(text)
            
            # 4. Naturalness Score (Composite Metric)
            # Logic: Humans have high perplexity + high burstiness + variable readability
            # We normalize these roughly to create a score.
            # (Simplified version for robustness)
            naturalness = (ppl * 0.4) + (burstiness * 0.4) + (flesch * 0.2)
            
            return {
                "perplexity": ppl,
                "burstiness": burstiness,
                "flesch_score": flesch,
                "grade_level": readability_grade,
                "word_count": word_count,
                "naturalness_score": naturalness
            }
        except Exception as e:
            print(f"Error processing text: {e}")
            return None

# Initialize Extractor
extractor = FeatureExtractor(device=device)

Loading GPT-2 model...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [7]:
# Safer loop that saves progress every 10,000 rows
import os

output_file = 'member1_features_matrix_partial.csv'
batch_size = 10000
features_list = []

print("Starting Feature Extraction with Auto-Save...")

for index, row in tqdm(df.iterrows(), total=df.shape[0]):
    text = row['text']
    features = extractor.extract_features(text)
    
    if features:
        features['generated'] = row['generated'] 
        features['original_text_idx'] = index
        features_list.append(features)
    
    # Save every 10,000 rows
    if len(features_list) >= batch_size:
        temp_df = pd.DataFrame(features_list)
        # If file doesn't exist, write with header. If it does, append without header.
        if not os.path.exists(output_file):
            temp_df.to_csv(output_file, index=False)
        else:
            temp_df.to_csv(output_file, mode='a', header=False, index=False)
        
        # Clear memory
        features_list = []
        print(f"Saved batch! Progress: {index}/{df.shape[0]}")

# Save any remaining rows at the end
if features_list:
    temp_df = pd.DataFrame(features_list)
    if not os.path.exists(output_file):
        temp_df.to_csv(output_file, index=False)
    else:
        temp_df.to_csv(output_file, mode='a', header=False, index=False)

print("Final Extraction Complete! Download 'member1_features_matrix_partial.csv'.")

Starting Feature Extraction with Auto-Save...


  3%|▎         | 10001/290888 [08:30<4:18:51, 18.09it/s]

Saved batch! Progress: 9999/290888


  7%|▋         | 20004/290888 [16:58<3:41:13, 20.41it/s] 

Saved batch! Progress: 19999/290888


...

Saved batch! Progress: 279999/290888


100%|█████████▉| 290000/290888 [4:09:32<00:47, 18.83it/s]

Saved batch! Progress: 289999/290888


100%|██████████| 290888/290888 [4:10:19<00:00, 19.37it/s]

Final Extraction Complete! Download 'member1_features_matrix_partial.csv'.


In [ ]:
# Save to CSV
features_df.to_csv('member1_features_matrix.csv', index=False)
print("File 'member1_features_matrix.csv' saved successfully.")